# Phase 3: Error Analysis

This is the part of the project that actually matters. A macro F1 number alone says almost nothing about whether a model is clinically useful, or which of its mistakes are forgivable. This notebook investigates the transformer trained in Phase 2 along four axes:

1. **Confusion patterns** — which stages get mistaken for which, and why (single-channel EEG has real, known limits here).
2. **Transition errors** — does the model do worse right around a real stage change?
3. **Subject variability** — are some subjects systematically harder, and what does a bad night look like?
4. **Confidence calibration** — when the model is wrong, does it at least know it?

The CNN baseline's Phase 2 numbers are referenced for context throughout, but the transformer is the primary subject of this analysis, since it's the model the project's research question is actually about.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
from torch.utils.data import DataLoader

from src.analysis import compute_transition_mask, per_subject_f1, top_confusion_pairs
from src.baseline import CNNBaseline
from src.config import BATCH_SIZE, CASSETTE_DIR, FIGURES_DIR, LABEL_NAMES, get_device, set_seed
from src.data import (
    SleepEDFDataset,
    build_epoch_arrays,
    discover_subjects,
    get_subject_split,
)
from src.train import evaluate, train_model
from src.transformer import TransformerClassifier

set_seed()
device = get_device()
FIGURES_DIR.mkdir(exist_ok=True)
print(f"device: {device}")

## Setup: rebuild the split, train both models

Same subject-level split and training regime as Phases 1-2, so results here are directly comparable to the head-to-head table in `02_transformer.ipynb`.

In [ ]:
subjects = discover_subjects(CASSETTE_DIR)
subject_ids = [s.subject_id for s in subjects]
train_ids, val_ids, test_ids = get_subject_split(subject_ids)

train_epochs, train_labels, _ = build_epoch_arrays(subjects, train_ids)
val_epochs, val_labels, _ = build_epoch_arrays(subjects, val_ids)
test_epochs, test_labels, test_subject_ids = build_epoch_arrays(subjects, test_ids)

train_loader = DataLoader(SleepEDFDataset(train_epochs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepEDFDataset(val_epochs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SleepEDFDataset(test_epochs, test_labels), batch_size=BATCH_SIZE, shuffle=False)

print(f"test subjects: {test_ids}  ({test_epochs.shape[0]} epochs)")

In [ ]:
set_seed()
baseline_model = CNNBaseline()
baseline_model, _ = train_model(baseline_model, train_loader, val_loader, device=device)
baseline_results = evaluate(baseline_model, test_loader, device=device)
baseline_f1 = f1_score(baseline_results["labels"], baseline_results["preds"], average="macro")

set_seed()
transformer_model = TransformerClassifier()
transformer_model, _ = train_model(transformer_model, train_loader, val_loader, device=device)
results = evaluate(transformer_model, test_loader, device=device)
transformer_f1 = f1_score(results["labels"], results["preds"], average="macro")

preds, labels, probs = results["preds"], results["labels"], results["probs"]

print(f"CNN baseline macro F1: {baseline_f1:.4f}")
print(f"Transformer macro F1:  {transformer_f1:.4f}")

## 1. Confusion patterns

Which stages does the transformer actually confuse, and does it match what single-channel EEG (no EOG, no EMG) would predict? The clinical R&K/AASM scoring rules lean on eye movements to separate REM from N1 and muscle tone to confirm Wake — signals this project deliberately excludes to keep the input simple. Any REM<->N1 or REM<->W confusion is at least partly a consequence of that choice, not necessarily a transformer weakness.

In [ ]:
cm = confusion_matrix(labels, preds, normalize="true")
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("Transformer: normalized confusion matrix (test set)")
plt.show()

In [ ]:
HYPOTHESES = {
    ("N1", "W"): "both show low-amplitude, mixed-frequency EEG; without EOG/EMG, slow eye movements at sleep onset look like wakefulness.",
    ("W", "N1"): "the reverse of N1<->W: drowsy wakefulness and true N1 overlap heavily in raw EEG amplitude and frequency content.",
    ("N1", "REM"): "both are low-amplitude, mixed-frequency stages; the clinical distinction leans heavily on EOG (rapid eye movements) and EMG (muscle atonia), neither of which this single-channel model has access to.",
    ("REM", "N1"): "see N1<->REM -- this is a known, expected confusion pair for single-channel EEG.",
    ("N2", "N3"): "N2/N3 are separated by a slow-wave-percentage threshold (roughly 20% delta activity); epochs straddling that threshold are borderline by construction, not just for the model.",
    ("N3", "N2"): "see N2<->N3 -- borderline slow-wave percentage near the scoring threshold.",
    ("N2", "W"): "less expected; may indicate brief arousals within N2 that share some spectral content with light wakefulness.",
    ("W", "N2"): "less expected; check whether these epochs sit near recording artefacts or the crop boundary.",
}
DEFAULT_HYPOTHESIS = "no strong prior hypothesis for this pair -- worth a manual look at a few example epochs."

pairs = top_confusion_pairs(labels, preds, LABEL_NAMES, top_k=3)
print("Top 3 confusion pairs (true -> predicted, count):\n")
for true_name, pred_name, count in pairs:
    hypothesis = HYPOTHESES.get((true_name, pred_name), DEFAULT_HYPOTHESIS)
    print(f"  {true_name} -> {pred_name}  ({count} epochs)")
    print(f"    hypothesis: {hypothesis}\n")

## 2. Transition errors

Sleep staging is scored epoch-by-epoch, but sleep itself is continuous -- an epoch that straddles a real stage change is inherently more ambiguous than one deep in a stable stretch. This checks whether the model's error rate reflects that: an epoch counts as "near transition" if the epoch immediately before or after it (within the same subject's recording) carries a different true label.

In [ ]:
transition_mask = compute_transition_mask(labels, test_subject_ids)

n_transition = transition_mask.sum()
n_stable = (~transition_mask).sum()
f1_transition = f1_score(labels[transition_mask], preds[transition_mask], average="macro") if n_transition else float("nan")
f1_stable = f1_score(labels[~transition_mask], preds[~transition_mask], average="macro") if n_stable else float("nan")

print(f"near-transition epochs: {n_transition}  (macro F1: {f1_transition:.4f})")
print(f"stable epochs:          {n_stable}  (macro F1: {f1_stable:.4f})")

if n_transition and n_stable:
    gap = f1_stable - f1_transition
    direction = "worse" if gap > 0 else "better (or no worse)"
    print(f"\nmodel performs {direction} near transitions (stable - transition F1 = {gap:+.4f})")
    print("this matches the standard finding in the sleep-staging literature" if gap > 0
          else "this does NOT match the standard literature finding -- worth double-checking the transition-mask logic before trusting it")

## 3. Subject variability

A single aggregate F1 can hide a model that does great on most subjects and badly on a few. This computes F1 separately per test subject.

In [ ]:
subject_f1 = per_subject_f1(labels, preds, test_subject_ids)
sorted_subjects = sorted(subject_f1.items(), key=lambda kv: kv[1])

print("per-subject macro F1 (sorted, hardest first):")
for subject, f1 in sorted_subjects:
    print(f"  subject {subject}: {f1:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([s for s, _ in sorted_subjects], [f1 for _, f1 in sorted_subjects], color="steelblue")
ax.set_xlabel("subject")
ax.set_ylabel("macro F1")
ax.set_title("Per-subject macro F1 (test set)")
ax.axhline(transformer_f1, color="gray", linestyle="--", linewidth=1, label="overall macro F1")
ax.legend()
plt.show()

**Figure: predicted vs. true hypnogram for the hardest test subject(s).** A hypnogram plots sleep stage against time across the whole night; the true trace and the model's predicted trace are overlaid so disagreements are visually obvious. Stages are ordered top-to-bottom as W, REM, N1, N2, N3 -- the conventional clinical hypnogram layout, which places REM just below Wake to reflect its "paradoxical" physiology (an active brain in an atonic body) rather than ordering strictly by depth of sleep. Look for whether errors are scattered randomly or clustered in specific stretches of the night (e.g. a long run of misclassified light sleep) -- clustering suggests a systematic issue (e.g. a noisy electrode for part of the night) rather than epoch-by-epoch noise.

In [ ]:
PLOT_ORDER = ["W", "REM", "N1", "N2", "N3"]
label_id_to_plot_pos = {LABEL_NAMES.index(name): pos for pos, name in enumerate(PLOT_ORDER)}

n_hardest = min(3, len(sorted_subjects))
hardest_subjects = [s for s, _ in sorted_subjects[:n_hardest]]

fig, axes = plt.subplots(n_hardest, 1, figsize=(12, 2.5 * n_hardest), sharex=False, squeeze=False)
axes = axes[:, 0]

for ax, subject in zip(axes, hardest_subjects):
    idx = np.where(test_subject_ids == subject)[0]
    true_seq = [label_id_to_plot_pos[l] for l in labels[idx]]
    pred_seq = [label_id_to_plot_pos[p] for p in preds[idx]]
    time_hours = np.arange(len(idx)) * 30 / 3600

    ax.step(time_hours, true_seq, where="post", label="true", color="black", linewidth=1.2)
    ax.step(time_hours, pred_seq, where="post", label="predicted", color="crimson", linewidth=1.0, alpha=0.75)
    ax.set_yticks(range(len(PLOT_ORDER)))
    ax.set_yticklabels(PLOT_ORDER)
    ax.invert_yaxis()
    ax.set_ylabel("stage")
    ax.set_title(f"subject {subject}  (macro F1: {subject_f1[subject]:.4f})")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("time (hours)")
fig.tight_layout()
plt.show()

## 4. Confidence calibration

For a model to be useful as a decision-support tool (rather than a black box), it should ideally be less confident when it's wrong. This compares the model's max softmax probability on correct vs. incorrect predictions.

In [ ]:
max_prob = probs.max(axis=1)
correct_mask = preds == labels

print(f"mean max-prob, correct predictions:   {max_prob[correct_mask].mean():.4f}")
print(f"mean max-prob, incorrect predictions: {max_prob[~correct_mask].mean():.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 1, 30)
ax.hist(max_prob[correct_mask], bins=bins, alpha=0.6, label="correct", color="seagreen", density=True)
ax.hist(max_prob[~correct_mask], bins=bins, alpha=0.6, label="incorrect", color="crimson", density=True)
ax.set_xlabel("max softmax probability")
ax.set_ylabel("density")
ax.set_title("Confidence distribution: correct vs. incorrect predictions")
ax.legend()
plt.show()

## Summary figure

**Figure: `figures/error_analysis_grid.png`, a 2x2 panel summarizing the four analyses above.** Top-left: confusion matrix. Top-right: macro F1 near stage transitions vs. in stable stretches. Bottom-left: per-subject macro F1. Bottom-right: confidence calibration (correct vs. incorrect). This is the single figure to pull into the blog post.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(ax=axes[0, 0], cmap="Blues", values_format=".2f", colorbar=False)
axes[0, 0].set_title("Confusion matrix")

axes[0, 1].bar(["near transition", "stable"], [f1_transition, f1_stable], color=["crimson", "steelblue"])
axes[0, 1].set_ylabel("macro F1")
axes[0, 1].set_title("F1: near-transition vs. stable")

axes[1, 0].bar([s for s, _ in sorted_subjects], [f1 for _, f1 in sorted_subjects], color="steelblue")
axes[1, 0].set_xlabel("subject")
axes[1, 0].set_ylabel("macro F1")
axes[1, 0].set_title("Per-subject macro F1")

axes[1, 1].hist(max_prob[correct_mask], bins=bins, alpha=0.6, label="correct", color="seagreen", density=True)
axes[1, 1].hist(max_prob[~correct_mask], bins=bins, alpha=0.6, label="incorrect", color="crimson", density=True)
axes[1, 1].set_xlabel("max softmax probability")
axes[1, 1].set_ylabel("density")
axes[1, 1].set_title("Confidence calibration")
axes[1, 1].legend()

fig.suptitle("Transformer error analysis: Sleep-EDF test set", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "error_analysis_grid.png", dpi=150)
plt.show()

print(f"saved to {FIGURES_DIR / 'error_analysis_grid.png'}")

## Next steps

Findings from this notebook are summarized in plain language in `results.md`, alongside the project's honest self-assessment: what this establishes, what it doesn't, and what a real extension would need (multi-channel input, sequence-of-epochs context, external validation).